In [1]:
import polars as pl

In [2]:
markets = pl.read_parquet('data/processed/markets_political.parquet')

In [3]:
m_list = markets['condition_id'].to_list()

In [4]:
df = (
    pl.scan_parquet('data/raw/quant.parquet', parallel='prefiltered')
    .filter(pl.col('condition_id').str.contains_any(m_list))
    .collect()
)

In [6]:
df.write_parquet('data/processed/quant-filtered.parquet')

In [8]:
print(df.shape)
print(df.columns)
print(df.dtypes)

# Date range
print(df['timestamp'].min())
print(df['timestamp'].max())

# Unique markets captured
print(f"Unique markets: {df['condition_id'].n_unique()}")
print(f"Expected: {len(m_list)}")

# Trades per market distribution
print(df.group_by('condition_id').len()['len'].describe())

(31722655, 13)
['timestamp', 'block_number', 'transaction_hash', 'log_index', 'market_id', 'condition_id', 'event_id', 'price', 'usd_amount', 'token_amount', 'side', 'maker', 'taker']
[UInt64, UInt64, String, UInt32, String, String, String, Float64, Float64, Float64, String, String, String]
1669060169
1773657371
Unique markets: 20111
Expected: 20689
shape: (9, 2)
┌────────────┬──────────────┐
│ statistic  ┆ value        │
│ ---        ┆ ---          │
│ str        ┆ f64          │
╞════════════╪══════════════╡
│ count      ┆ 20111.0      │
│ null_count ┆ 0.0          │
│ mean       ┆ 1577.3783    │
│ std        ┆ 25077.087049 │
│ min        ┆ 1.0          │
│ 25%        ┆ 72.0         │
│ 50%        ┆ 218.0        │
│ 75%        ┆ 630.0        │
│ max        ┆ 2.749424e6   │
└────────────┴──────────────┘


In [9]:
import datetime
print(datetime.datetime.fromtimestamp(1669060169))  # earliest
print(datetime.datetime.fromtimestamp(1773657371))  # latest

2022-11-21 19:49:29
2026-03-16 10:36:11


In [10]:
missing = set(m_list) - set(df['condition_id'].to_list())
missing_markets = markets.filter(pl.col('condition_id').is_in(list(missing)))
print(missing_markets['resolved_yes'].value_counts())
print(missing_markets.select(['question']).sample(5))

shape: (3, 2)
┌──────────────┬───────┐
│ resolved_yes ┆ count │
│ ---          ┆ ---   │
│ bool         ┆ u32   │
╞══════════════╪═══════╡
│ false        ┆ 361   │
│ true         ┆ 216   │
│ null         ┆ 1     │
└──────────────┴───────┘
shape: (5, 1)
┌─────────────────────────────────┐
│ question                        │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ Cedric Coward: Points Over 13.… │
│ Isaiah Stewart: Rebounds Over … │
│ Who will have a higher net wor… │
│ Will Josh Mandel win the Repub… │
│ Will President Biden mention M… │
└─────────────────────────────────┘


In [11]:
df = df.with_columns(
    pl.from_epoch(pl.col('timestamp'), time_unit='s').alias('datetime')
).with_columns(
    pl.col('datetime').dt.convert_time_zone('UTC').dt.cast_time_unit('ms')
)
print(df['datetime'].min())
print(df['datetime'].max())

2022-11-21 19:49:29+00:00
2026-03-16 10:36:11+00:00


## Feature Matrix Analysis

In [2]:
import polars as pl

In [3]:
df = pl.read_parquet('data/processed/feature_matrix.parquet')

In [4]:
df

market_id,resolved_yes,price_start,price_end,price_mean,price_min,price_max,price_volatility,price_range,price_momentum,log_total_volume,log_trade_count,log_avg_trade_size,buy_ratio,log_market_volume,days_active,days_to_resolution
str,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
"""0x966d5b4f4f52b7a4bcdff33cc5d9…",false,0.2,0.011,0.122549,0.01,0.28,0.102934,0.27,-0.189,3.952013,3.465736,0.97322,0.16129,6.821727,14,0
"""0x646f5168c9effb4abe921e31ab37…",false,0.24,0.14,0.206471,0.14,0.27,0.03639,0.13,-0.1,5.074486,2.890372,2.336645,0.294118,6.679218,6,0
"""0x94fe4f0bfd01584ebf11b9fad940…",false,0.81,0.36,0.513,0.31,0.81,0.183488,0.5,-0.45,4.119525,2.397895,1.953453,0.8,7.690729,6,0
"""0xa2399fd5143ebb08a0cfe95c18bf…",false,0.21,0.16,0.345385,0.12,0.65,0.192296,0.53,-0.05,3.971235,3.295837,1.099253,0.538462,7.490785,6,0
"""0x5e6927c17386d5e4225af94c495c…",false,0.36,0.45,0.482,0.36,0.52,0.046619,0.16,0.09,4.738039,2.397895,2.511305,0.6,6.699553,6,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""0x6ed4632467b72f99b227b1a42fa3…",false,0.002,0.001,0.00213,0.001,0.037,0.003174,0.036,-0.001,3.293612,5.602119,0.091735,0.07037,10.10296,85,8
"""0x735f025c4d6d5a64b74078c11a12…",false,0.002,0.001,0.002758,0.001,0.038,0.004365,0.037,-0.001,7.234278,5.805135,1.645714,0.132931,10.304549,85,2
"""0x46bf30bed377ca32f5cba14cfbe8…",false,0.001,0.001,0.001054,0.001,0.002,0.000227,0.001,0.0,2.948116,5.627621,0.063195,0.054152,9.940927,85,9


In [5]:
feature_matrix = pl.read_parquet('data/processed/feature_matrix.parquet')

print(feature_matrix.shape)
print(feature_matrix.null_count())
print(feature_matrix.describe())
print(feature_matrix['resolved_yes'].value_counts())
print(feature_matrix.head(5))

(14443, 17)
shape: (1, 17)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ market_id ┆ resolved_ ┆ price_sta ┆ price_end ┆ … ┆ buy_ratio ┆ log_marke ┆ days_acti ┆ days_to_ │
│ ---       ┆ yes       ┆ rt        ┆ ---       ┆   ┆ ---       ┆ t_volume  ┆ ve        ┆ resoluti │
│ u32       ┆ ---       ┆ ---       ┆ u32       ┆   ┆ u32       ┆ ---       ┆ ---       ┆ on       │
│           ┆ u32       ┆ u32       ┆           ┆   ┆           ┆ u32       ┆ u32       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ u32      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 0         ┆ 0         ┆ 0         ┆ 0         ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0        │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴──────────┘
shape: (9, 18)
┌───────────┬───────────┬───────────┬───────────┬

In [6]:
markets = pl.read_parquet('data/processed/markets_political.parquet')

feature_matrix = feature_matrix.join(
    markets.select(['condition_id', 'end_date']),
    left_on='market_id',
    right_on='condition_id',
    how='left'
)

In [8]:
feature_matrix.write_parquet('data/processed/feature_matrix.parquet')

In [9]:
# What is the correlation between price_end and resolved_yes?
import numpy as np
test_df = pl.read_parquet('data/model/test.parquet')
correlation = np.corrcoef(
    test_df['price_end'].to_numpy(),
    test_df['resolved_yes'].cast(pl.Int32).to_numpy()
)[0,1]
print(f"price_end vs resolved_yes correlation: {correlation:.4f}")

price_end vs resolved_yes correlation: 0.8399
